# IOAI — 2025 Contest Intent Slot Filling (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, glob, zipfile
os.makedirs('data', exist_ok=True)
os.system('pip -q install -U transformers sentence-transformers')
if not os.path.exists('data/train.conll'):
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_files('neoai-2025-intent-detection-and-slot-filling', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
print('데이터 준비:', 'data/train.conll' if os.path.exists('data/train.conll') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# NeoAI 2025 — Intent Detection & Slot Filling (교차언어)

> **Northern Eurasia Olympiad in AI 2025 · Kaggle playground competition (NLP · 결합 NLU)**

발화의 **의도(intent)** 분류 + **슬롯(slot) 채우기**(BIO 토큰 태깅)를 동시에 합니다. **train=영어**,
**validation/test=러시아어** (교차언어 제로샷). validation 375문장에 정답이 있어 **검증셋으로 로컬 채점**합니다.

**채점**: `submission.json`(문장별 `{"intent":..., "tags":[BIO,...]}`, validation 순서) → 점수 = **intent 정확도와
slot 스팬 F1의 평균**(이 사이트 자체 지표). 캐글 리더보드 1위 ≈0.586(캐글의 다른 지표).

⚠️ **아래 베이스라인 = 다국어 문장임베딩으로 intent 만 분류(슬롯은 전부 O)**: intent_acc ≈0.90 이지만 슬롯 F1=0 →
지표 ≈ **0.45**. 슬롯까지 예측하려면 토큰 분류 모델이 필요합니다(모범답안).

In [ ]:
import os, json, numpy as np, torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)
def parse_conll(path):
    items, toks, tags, intent = [], [], [], None
    for line in open(path, encoding='utf-8'):
        line = line.rstrip('\n')
        if line.startswith('# intent'): intent = line.split('=' if '=' in line else ':',1)[1].strip()
        elif line.startswith('#'): continue
        elif not line.strip():
            if toks: items.append({'tokens':toks,'intent':intent,'tags':tags}); toks,tags,intent=[],[],None
        else:
            p=line.split('\t')
            if len(p)>=4: toks.append(p[1]); tags.append(p[3])
    if toks: items.append({'tokens':toks,'intent':intent,'tags':tags})
    return items
tr = parse_conll('data/train.conll'); va = parse_conll('data/validation.conll')
intents = sorted(set(x['intent'] for x in tr))
print('train', len(tr), '| val(RU)', len(va), '| intents', len(intents))

In [ ]:
# BASELINE: 다국어 문장임베딩 최근접 centroid 로 intent (영어 centroid ← 러시아어 쿼리, 교차언어 정렬), 슬롯=전부 O
from sentence_transformers import SentenceTransformer
st = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device=dev)
tre = st.encode([' '.join(x['tokens']) for x in tr], batch_size=256, normalize_embeddings=True, show_progress_bar=False)
cent = np.stack([tre[[i for i,x in enumerate(tr) if x['intent']==it]].mean(0) for it in intents])
cent /= np.linalg.norm(cent, axis=1, keepdims=True)
vae = st.encode([' '.join(x['tokens']) for x in va], batch_size=256, normalize_embeddings=True, show_progress_bar=False)
pred_int = [intents[i] for i in (vae @ cent.T).argmax(1)]
sub = [{'intent': pi, 'tags': ['O']*len(x['tags'])} for pi, x in zip(pred_int, va)]
json.dump(sub, open('submission.json','w'))
print('wrote submission.json', len(sub), '(intent만 — 슬롯 토큰 분류로 개선하세요: 모범답안)')


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)